# Project A: Structural Analysis of the FlyWire Whole-Brain Connectome

**Data**: FlyWire v783 snapshot — Zenodo:10676866  
**Goal**: Quantify structural properties of the adult *Drosophila* sensorimotor connectome  
and translate them into inductive biases for a GNN controller (Project B).

## Analyses
1. Load data, explore schema, basic counts
2. Build directed graph, full-graph stats (degree distribution, assortativity)
3. Classify neurons: sensor / inter / motor via neuropil projection profiles
4. Feedforward / feedback / lateral weight balance
5. Hub detection — rich-club organisation
6. Betweenness centrality on sensorimotor subgraph
7. Community structure (Louvain)
8. Global hierarchy score (GRC)
9. Controller-prior summary

## Setup
```bash
pip install jupyter scipy
# Data in:  ../data/real/
#   proofread_connections_783.feather       (852 MB)
#   per_neuron_neuropil_count_pre_783.feather  (17 MB)
#   per_neuron_neuropil_count_post_783.feather (234 MB)
#   proofread_root_ids_783.npy              (1 MB)
```

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import networkx.algorithms.community as nxc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

DATA = Path('../data/real')
print('networkx:', nx.__version__)
print('pandas:  ', pd.__version__)
print('data dir exists:', DATA.exists())

---
## 1. Load Data & Explore Schema

In [ ]:
# Load connections (852 MB — takes ~30 s)
print('Loading connections...')
conn = pd.read_feather(DATA / 'proofread_connections_783.feather')
print(f'Connections table: {conn.shape[0]:,} rows × {conn.shape[1]} cols')
print(conn.dtypes)
conn.head(3)

In [ ]:
# Basic connection stats
print('=== Connection summary ===')
print(f'Unique pre-synaptic  neurons : {conn.iloc[:,0].nunique():>10,}')
print(f'Unique post-synaptic neurons : {conn.iloc[:,1].nunique():>10,}')
print(f'Total neuron pairs           : {len(conn):>10,}')

# Find the synapse count column (usually named 'syn_count', 'weight', 'n_syn', etc.)
syn_col = [c for c in conn.columns if any(k in c.lower() for k in ['syn','weight','count','n_'])]
print(f'\nSynapse count column candidates: {syn_col}')

# Display distribution of synapse counts
sc = conn[syn_col[0]] if syn_col else conn.iloc[:,2]
print(f'\nSynapse count per pair:')
print(f'  median : {sc.median():.0f}')
print(f'  mean   : {sc.mean():.1f}')
print(f'  max    : {sc.max():,}')
print(f'  >= 5   : {(sc >= 5).sum():,} pairs  ({100*(sc>=5).mean():.1f}%)')
print(f'  >= 3   : {(sc >= 3).sum():,} pairs  ({100*(sc>=3).mean():.1f}%)')

In [ ]:
# Load neuropil projection files
print('Loading neuropil files...')
np_pre  = pd.read_feather(DATA / 'per_neuron_neuropil_count_pre_783.feather')
np_post = pd.read_feather(DATA / 'per_neuron_neuropil_count_post_783.feather')

print(f'Neuropil PRE  table: {np_pre.shape}  — columns: {list(np_pre.columns[:6])}...')
print(f'Neuropil POST table: {np_post.shape} — columns: {list(np_post.columns[:6])}...')
np_pre.head(2)

---
## 2. Build Graph & Full-Graph Degree Statistics

Filter edges with synapse count ≥ 5 to remove weak/noisy connections.  
This is the threshold used in Lin et al. 2024.

In [ ]:
# ── Identify column names dynamically ──────────────────────────────
pre_col  = conn.columns[0]    # pre-synaptic neuron ID
post_col = conn.columns[1]    # post-synaptic neuron ID
syn_col  = conn.columns[2]    # synapse count / weight

print(f'Using columns: pre={pre_col!r}, post={post_col!r}, weight={syn_col!r}')

# ── Filter ──────────────────────────────────────────────────────────
SYN_THRESH = 5
edges = conn[conn[syn_col] >= SYN_THRESH][[pre_col, post_col, syn_col]].copy()
edges.columns = ['pre', 'post', 'weight']
print(f'Edges after filtering (≥{SYN_THRESH} synapses): {len(edges):,}')
print(f'Unique neurons in filtered graph: {pd.concat([edges.pre, edges.post]).nunique():,}')

In [ ]:
# ── Build directed graph ────────────────────────────────────────────
# NOTE: building a networkx graph from ~millions of edges takes ~2–5 min
print('Building directed graph (this takes a few minutes)...')
G_full = nx.from_pandas_edgelist(
    edges, source='pre', target='post', edge_attr='weight',
    create_using=nx.DiGraph()
)
print(f'Graph: {G_full.number_of_nodes():,} nodes, {G_full.number_of_edges():,} edges')
print(f'Density: {nx.density(G_full):.2e}')

In [ ]:
# ── Degree distribution (fast on full graph) ─────────────────────────
in_deg_seq  = sorted([d for _, d in G_full.in_degree()],  reverse=True)
out_deg_seq = sorted([d for _, d in G_full.out_degree()], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, seq, label, color in zip(
    axes,
    [in_deg_seq, out_deg_seq],
    ['In-degree', 'Out-degree'],
    ['steelblue', 'darkorange']
):
    ax.hist(seq, bins=80, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel('Number of neurons')
    ax.set_title(f'{label} distribution (full graph, n={len(seq):,})')
    ax.set_yscale('log')
    ax.set_xscale('log')

plt.tight_layout()
plt.savefig('../data/real/fig1_degree_distribution.png', dpi=150)
plt.show()

print(f'In-degree  — median: {np.median(in_deg_seq):.0f}, mean: {np.mean(in_deg_seq):.1f}, max: {max(in_deg_seq)}')
print(f'Out-degree — median: {np.median(out_deg_seq):.0f}, mean: {np.mean(out_deg_seq):.1f}, max: {max(out_deg_seq)}')

In [ ]:
# ── Weighted degree assortativity ──────────────────────────────────
# Do highly-connected neurons tend to connect to each other (rich-club tendency)?
r = nx.degree_assortativity_coefficient(G_full)
print(f'Degree assortativity coefficient: {r:.4f}')
print('  > 0 → hubs connect to hubs (assortative / rich-club tendency)')
print('  < 0 → hubs connect to low-degree nodes (disassortative)')

---
## 3. Neuron Type Classification: Sensor / Inter / Motor

We use the neuropil projection profiles to assign a functional layer to each neuron.

**Logic**:
- A neuron with mostly *post*-synaptic contacts in known **sensory input** neuropils → `sensor`
- A neuron with mostly *pre*-synaptic contacts in known **motor output** neuropils → `motor`  
- Everything else → `inter`

Known sensory input neuropils (Drosophila brain): `AL` (olfactory), `ME/LO/LOP` (visual),  
`AMMC/JO` (antennal mechanosensory), `PRW` (prow, mechanosensory).  
Known motor/descending output: columns containing `VNC`, `GNG`, `PS`, `SAD`.

In [ ]:
# ── Inspect neuropil column names ──────────────────────────────────
id_col_pre  = np_pre.columns[0]    # root_id / neuron ID column
id_col_post = np_post.columns[0]
neuropil_cols_pre  = [c for c in np_pre.columns  if c != id_col_pre]
neuropil_cols_post = [c for c in np_post.columns if c != id_col_post]

print('Neuropil columns (pre):', neuropil_cols_pre[:20], '...')
print('Neuropil columns (post):', neuropil_cols_post[:20], '...')
print(f'\nTotal neuropils in pre file:  {len(neuropil_cols_pre)}')
print(f'Total neuropils in post file: {len(neuropil_cols_post)}')

In [ ]:
# ── Define sensory and motor neuropil keywords ─────────────────────
# Adjust these lists if column names differ — run the cell above first
SENSORY_KEYWORDS = ['AL', 'ME', 'LO', 'LOP', 'AMMC', 'JO', 'PRW', 'AOTU', 'BU', 'EB', 'FB']
MOTOR_KEYWORDS   = ['VNC', 'GNG', 'PS', 'SAD', 'NP', 'WED']

def find_cols(col_list, keywords):
    return [c for c in col_list if any(k in c.upper() for k in keywords)]

sensory_neuropil_cols_post = find_cols(neuropil_cols_post, SENSORY_KEYWORDS)
motor_neuropil_cols_pre    = find_cols(neuropil_cols_pre,  MOTOR_KEYWORDS)

print(f'Sensory post-synaptic neuropil columns ({len(sensory_neuropil_cols_post)}):',
      sensory_neuropil_cols_post[:10])
print(f'Motor pre-synaptic neuropil columns ({len(motor_neuropil_cols_pre)}):',
      motor_neuropil_cols_pre[:10])

In [ ]:
# ── Compute per-neuron sensory/motor scores ─────────────────────────
np_post_indexed = np_post.set_index(id_col_post)
np_pre_indexed  = np_pre.set_index(id_col_pre)

# Sensory score = fraction of post-synaptic inputs from sensory neuropils
post_total  = np_post_indexed[neuropil_cols_post].sum(axis=1).replace(0, np.nan)
sens_score  = np_post_indexed[sensory_neuropil_cols_post].sum(axis=1) / post_total

# Motor score = fraction of pre-synaptic outputs into motor neuropils
pre_total   = np_pre_indexed[neuropil_cols_pre].sum(axis=1).replace(0, np.nan)
motor_score = np_pre_indexed[motor_neuropil_cols_pre].sum(axis=1) / pre_total

# ── Assign types ────────────────────────────────────────────────────
SENSOR_THRESH = 0.30   # ≥30% inputs from sensory neuropils
MOTOR_THRESH  = 0.30   # ≥30% outputs into motor neuropils

all_ids = pd.Index(G_full.nodes)

is_sensor = sens_score.reindex(all_ids).fillna(0) >= SENSOR_THRESH
is_motor  = motor_score.reindex(all_ids).fillna(0) >= MOTOR_THRESH

# Priority: sensor > motor > inter (a neuron can't be both; sensory takes priority)
node_type = pd.Series('inter', index=all_ids)
node_type[is_motor]  = 'motor'
node_type[is_sensor] = 'sensor'

type_counts = node_type.value_counts()
print('Neuron type distribution:')
for t, c in type_counts.items():
    print(f'  {t:<8}: {c:>7,}  ({100*c/len(node_type):.1f}%)')

In [ ]:
# ── Degree distribution split by type ──────────────────────────────
LAYER_COLOR = {'sensor': '#4c9be8', 'inter': '#f5a623', 'motor': '#4caf50'}

in_deg_map  = dict(G_full.in_degree())
out_deg_map = dict(G_full.out_degree())

node_df = pd.DataFrame({
    'id':         all_ids,
    'type':       node_type.values,
    'in_degree':  [in_deg_map.get(n, 0) for n in all_ids],
    'out_degree': [out_deg_map.get(n, 0) for n in all_ids],
})
node_df['total_degree'] = node_df['in_degree'] + node_df['out_degree']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col in zip(axes, ['in_degree', 'out_degree']):
    for t, grp in node_df.groupby('type'):
        vals = grp[col].clip(upper=grp[col].quantile(0.99))
        ax.hist(vals, bins=50, alpha=0.6, color=LAYER_COLOR[t], label=t, density=True)
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Density')
    ax.set_title(col.replace('_', ' ').title() + ' by Neuron Type')
    ax.set_yscale('log')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/real/fig2_degree_by_type.png', dpi=150)
plt.show()

print(node_df.groupby('type')[['in_degree', 'out_degree', 'total_degree']].describe().round(1))

---
## 4. Feedforward / Feedback / Lateral Balance

Classify every edge by the relative layer of its endpoints:
- **Feedforward**: sensor→inter, inter→motor, sensor→motor
- **Feedback**: motor→inter, inter→sensor, motor→sensor  
- **Lateral**: same-type connections (inter→inter, sensor→sensor, motor→motor)

In [ ]:
LAYER_RANK = {'sensor': 0, 'inter': 1, 'motor': 2}

type_dict = node_type.to_dict()

ff_w = fb_w = lat_w = 0.0
ff_n = fb_n = lat_n = 0

for u, v, d in G_full.edges(data=True):
    w   = d.get('weight', 1.0)
    r_u = LAYER_RANK.get(type_dict.get(u, 'inter'), 1)
    r_v = LAYER_RANK.get(type_dict.get(v, 'inter'), 1)
    if   r_v > r_u: ff_w  += w; ff_n  += 1
    elif r_v < r_u: fb_w  += w; fb_n  += 1
    else:           lat_w += w; lat_n += 1

total_w = ff_w + fb_w + lat_w
total_n = ff_n + fb_n + lat_n

results = {
    'feedforward': (ff_w,  ff_n),
    'feedback':    (fb_w,  fb_n),
    'lateral':     (lat_w, lat_n),
}

print('Edge direction balance (by synapse weight / by edge count):')
for label, (w, n) in results.items():
    print(f'  {label:<14}: {100*w/total_w:5.1f}% weight  |  {100*n/total_n:5.1f}% edges')

labels = list(results.keys())
w_vals = [results[k][0] for k in labels]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, w_vals, color=['cornflowerblue', 'tomato', 'goldenrod'])
ax.set_ylabel('Total synaptic weight')
ax.set_title('FF / FB / Lateral weight balance (real connectome)')
for bar, w in zip(bars, w_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{100*w/total_w:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../data/real/fig3_ff_fb_balance.png', dpi=150)
plt.show()

---
## 5. Rich-Club Organisation

The rich-club coefficient φ(k) measures whether high-degree nodes (degree > k) connect to each other  
more densely than expected by chance. Values > 1.0 vs a null model indicate rich-club structure.

Lin et al. 2024 found that ~30% of FlyWire neurons form a rich-club hub layer.

In [ ]:
# Rich-club requires undirected graph
G_undirected = G_full.to_undirected()

# Compute rich-club coefficient for a range of degree thresholds
# NOTE: this can take 5–15 min on the full graph; reduce k_max if slow
print('Computing rich-club coefficient (may take several minutes)...')
rc = nx.rich_club_coefficient(G_undirected, normalized=False)

# Plot
k_vals = sorted(rc.keys())
rc_vals = [rc[k] for k in k_vals]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_vals, rc_vals, color='purple', lw=1.5)
ax.set_xlabel('Degree threshold k')
ax.set_ylabel('Rich-club coefficient φ(k)')
ax.set_title('Rich-club organisation of the FlyWire connectome')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('../data/real/fig4_rich_club.png', dpi=150)
plt.show()

# Hub identification: top 20% by total degree
inter_df = node_df[node_df['type'] == 'inter'].copy()
hub_threshold = inter_df['total_degree'].quantile(0.80)
hubs = inter_df[inter_df['total_degree'] >= hub_threshold]
print(f'\nHub threshold (80th percentile degree, inter only): {hub_threshold:.0f}')
print(f'Hub neurons: {len(hubs):,}  ({100*len(hubs)/len(inter_df):.1f}% of inter-neurons)')

---
## 6. Betweenness Centrality — Bottleneck Detection

We work on a **sensorimotor subgraph** to make computation tractable (~few thousand nodes).  
The subgraph contains: all sensor neurons + all motor neurons + inter-neurons directly  
connecting them (2-hop neighbourhood).

In [ ]:
# ── Build sensorimotor subgraph ─────────────────────────────────────
sensor_ids = set(node_df[node_df['type'] == 'sensor']['id'])
motor_ids  = set(node_df[node_df['type'] == 'motor']['id'])

# Inter-neurons that have BOTH an upstream sensor connection AND a downstream motor connection
# = neurons sitting on a sensor→inter→motor path
inter_with_sensor_input = set()
inter_with_motor_output = set()

for u, v in G_full.edges():
    if u in sensor_ids and type_dict.get(v) == 'inter':
        inter_with_sensor_input.add(v)
    if type_dict.get(u) == 'inter' and v in motor_ids:
        inter_with_motor_output.add(u)

sm_inter = inter_with_sensor_input & inter_with_motor_output
sm_nodes = sensor_ids | motor_ids | sm_inter

G_sm = G_full.subgraph(sm_nodes).copy()
print(f'Sensorimotor subgraph: {G_sm.number_of_nodes():,} nodes, {G_sm.number_of_edges():,} edges')
print(f'  sensor : {len(sensor_ids & sm_nodes):,}')
print(f'  inter  : {len(sm_inter):,}')
print(f'  motor  : {len(motor_ids & sm_nodes):,}')

In [ ]:
# ── Approximate betweenness (k=500 samples) ─────────────────────────
# k=500 gives good approximation; reduce to k=100 if memory is tight
print('Computing approximate betweenness centrality (k=500)...')
bc = nx.betweenness_centrality(G_sm, normalized=True, weight='weight', k=500, seed=42)
print('Done.')

bc_series = pd.Series(bc)
bc_df = pd.DataFrame({
    'id':          bc_series.index,
    'betweenness': bc_series.values,
    'type':        [type_dict.get(n, 'inter') for n in bc_series.index]
})

# Top bottleneck neurons
print('\nTop 15 bottleneck neurons:')
print(bc_df.nlargest(15, 'betweenness')[['id', 'type', 'betweenness']].to_string(index=False))

# Plot betweenness by type
fig, ax = plt.subplots(figsize=(8, 4))
for t in ['sensor', 'inter', 'motor']:
    subset = bc_df[bc_df['type'] == t]['betweenness']
    ax.hist(subset, bins=40, alpha=0.7, color=LAYER_COLOR[t],
            label=f'{t} (n={len(subset):,})', density=True)
ax.set_xlabel('Betweenness centrality')
ax.set_ylabel('Density')
ax.set_title('Betweenness centrality distribution by type\n(sensorimotor subgraph)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('../data/real/fig5_betweenness.png', dpi=150)
plt.show()

---
## 7. Community Structure (Louvain)

Louvain algorithm detects densely-connected communities within the network.  
Communities in the inter-layer correspond to candidate functional sub-circuits —  
and potential sub-policy modules in a hierarchical GNN controller.

In [ ]:
# Run Louvain on sensorimotor subgraph (undirected version)
# NOTE: for very large subgraphs this can take 5–20 min
print('Running Louvain community detection on sensorimotor subgraph...')
G_sm_undirected = G_sm.to_undirected()
communities = nxc.louvain_communities(G_sm_undirected, seed=42, weight='weight')
print(f'Found {len(communities)} communities')

# Assign community IDs to nodes
node_to_comm = {}
for cid, comm in enumerate(communities):
    for node in comm:
        node_to_comm[node] = cid

# Community size distribution
comm_sizes = [len(c) for c in communities]
comm_sizes.sort(reverse=True)
print(f'\nCommunity sizes: min={min(comm_sizes)}, median={int(np.median(comm_sizes))}, max={max(comm_sizes)}')
print(f'Largest 5: {comm_sizes[:5]}')

# For each large community: what fraction is sensor/inter/motor?
print('\nType composition of 5 largest communities:')
for i, comm in enumerate(sorted(communities, key=len, reverse=True)[:5]):
    types = Counter(type_dict.get(n, 'inter') for n in comm)
    total = sum(types.values())
    print(f'  Community {i+1} (n={total}): '
          f'sensor={types.get("sensor",0)/total:.0%}, '
          f'inter={types.get("inter",0)/total:.0%}, '
          f'motor={types.get("motor",0)/total:.0%}')

In [ ]:
# Community size histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(comm_sizes, bins=40, color='mediumpurple', edgecolor='white')
ax.set_xlabel('Community size (number of neurons)')
ax.set_ylabel('Count')
ax.set_title(f'Louvain community size distribution ({len(communities)} communities)')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('../data/real/fig6_communities.png', dpi=150)
plt.show()

---
## 8. Global Hierarchy Score (GRC)

Global Reaching Centrality measures how hierarchically organised a network is.  
GRC = 1.0 → perfect DAG (pure feedforward);  GRC = 0.0 → no hierarchy (fully flat).

In [ ]:
# GRC on sensorimotor subgraph (faster than full graph)
# NOTE: this can take 5–30 min depending on subgraph size
print('Computing Global Reaching Centrality on sensorimotor subgraph...')
grc_sm = nx.global_reaching_centrality(G_sm, weight='weight', normalized=True)
print(f'GRC (sensorimotor subgraph): {grc_sm:.4f}')
print()
print('Interpretation:')
print(f'  GRC={grc_sm:.3f}: ', end='')
if grc_sm > 0.5:
    print('strongly hierarchical — fast reactive throughput, good for reflex-like control')
elif grc_sm > 0.2:
    print('moderately hierarchical — balanced reflex speed + temporal integration')
else:
    print('weakly hierarchical — rich recurrent dynamics, more like a reservoir')

---
## 9. Controller-Prior Summary

Translate structural measurements into architectural specifications for Project B.

In [ ]:
# ── Full summary ───────────────────────────────────────────────────
print('=' * 60)
print('PROJECT A — STRUCTURAL SUMMARY FOR PROJECT B')
print('=' * 60)

print(f"""
GRAPH SCALE
  Full graph     : {G_full.number_of_nodes():>8,} nodes, {G_full.number_of_edges():>10,} edges
  SM subgraph    : {G_sm.number_of_nodes():>8,} nodes, {G_sm.number_of_edges():>10,} edges
  Hub neurons    : {len(hubs):>8,} ({100*len(hubs)/len(inter_df):.1f}% of inter)

HIERARCHY
  GRC (SM)       : {grc_sm:.4f}  [0=flat, 1=DAG]
  FF fraction    : {100*ff_w/total_w:.1f}%  (by synaptic weight)
  FB fraction    : {100*fb_w/total_w:.1f}%
  Lateral frac.  : {100*lat_w/total_w:.1f}%

COMMUNITY
  Communities    : {len(communities)}
  Largest comm.  : {comm_sizes[0]} neurons

CONTROLLER SPECIFICATIONS FOR PROJECT B
  Message-passing depth   : 2 hops (sensor→inter→motor)
  Hub node hidden dim     : 2× wider than peripheral nodes
  Feedback edges          : keep as residual skip-connections
  Lateral recurrent edges : keep in adjacency (implicit memory)
  Community modules       : candidate modular sub-policies
""")
print('=' * 60)

In [ ]:
# ── Save summary figure ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel A: degree by type (in-degree)
ax = axes[0, 0]
for t, grp in node_df.groupby('type'):
    vals = grp['in_degree'].clip(upper=grp['in_degree'].quantile(0.99))
    ax.hist(vals, bins=50, alpha=0.6, color=LAYER_COLOR[t], label=t, density=True)
ax.set_title('A. In-degree by neuron type'); ax.set_yscale('log')
ax.set_xlabel('In-degree'); ax.legend()

# Panel B: FF/FB/lateral
ax = axes[0, 1]
bar_labels = ['Feedforward', 'Feedback', 'Lateral']
bar_vals   = [ff_w, fb_w, lat_w]
bars = ax.bar(bar_labels, bar_vals, color=['cornflowerblue', 'tomato', 'goldenrod'])
ax.set_title('B. FF / FB / Lateral balance')
ax.set_ylabel('Total synaptic weight')
for b, v in zip(bars, bar_vals):
    ax.text(b.get_x() + b.get_width()/2, b.get_height()*1.02,
            f'{100*v/total_w:.0f}%', ha='center')

# Panel C: betweenness by type
ax = axes[1, 0]
for t in ['sensor', 'inter', 'motor']:
    subset = bc_df[bc_df['type'] == t]['betweenness']
    ax.hist(subset, bins=40, alpha=0.7, color=LAYER_COLOR[t], label=t, density=True)
ax.set_title('C. Betweenness centrality (SM subgraph)')
ax.set_xlabel('Betweenness'); ax.set_yscale('log'); ax.legend()

# Panel D: community sizes
ax = axes[1, 1]
ax.hist(comm_sizes, bins=30, color='mediumpurple', edgecolor='white')
ax.set_title(f'D. Community sizes ({len(communities)} communities)')
ax.set_xlabel('Community size'); ax.set_xscale('log')

plt.suptitle('FlyWire Whole-Brain Connectome — Structural Analysis Summary', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../data/real/fig0_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved summary figure to ../data/real/fig0_summary.png')